# Preprocessing — Group 34

Continuation of the EDA notebook. Every preprocessing choice below is backed by code: an EDA finding (cited as EDA 1.x),
a check printed here, or a comparison on the validation folds.

**How choices are measured.** Each option is scored with the same five time-ordered folds (90-day purge) and the same
plain logistic regression, used only as a measuring instrument. We compare options **fold by fold** (paired difference in
ROC-AUC) and adopt a change only if its mean gain is larger than **two standard errors**; otherwise we keep the simpler
option, or decide on a stated principle. ROC-AUC is used because it does not depend on the 0.5 threshold, which is tuned
later, in the modelling notebook.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer   # noqa: F401 (needed before importing IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

DATA_DIR = Path("data")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)       # pandas/sklearn deprecation notices only
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # IterativeImputer early-stopping notice (step 5)
pd.set_option("display.width", 200)

# CSV Import

In [2]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["purchase_date"]).set_index("ID")
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["purchase_date"]).set_index("ID")
clientes_raw = pd.read_csv(DATA_DIR / "clientes.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

target = "repeat_purchase_90d"
print("test rows in sample_submission order:", test_raw.index.tolist() == sample["ID"].tolist())

test rows in sample_submission order: True


# Sorting by Date

- The TimeSeriesSplit needs the **training** rows in time order, so only `train` is sorted.
- `test` is **not** sorted: the submission must follow the row order of `sample_submission.csv`, and the model does not
  need test in time order to predict it.

In [3]:
train_raw = train_raw.sort_values(["purchase_date", "ID"])   # ID breaks ties on the same day
print("train sorted by date:", train_raw["purchase_date"].is_monotonic_increasing)

train sorted by date: False


# Quick EDA and Data Types

In [4]:
train_raw.head(3)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000812,55.86,51.99,51.99,3.87,4.39,0.00,0.0,28.00,1.0,1.0,...,0.0,120503,Maybe Tomorrow Payment 12,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 49,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-11,-0.012756
10007082,268.76,436.99,218.50,50.26,0.00,218.49,0.0,116.14,9.0,1.0,...,0.0,113110,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 80,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-11,-0.503701
10000232,541.92,508.20,508.20,33.72,0.00,0.00,0.0,285.70,6.0,1.0,...,0.0,568558,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 10,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-12,1.496488


In [5]:
print(train_raw.shape)
train_raw.isna().mean().round(4).to_frame("share missing").T

(6534, 38)


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
share missing,0.0099,0.0099,0.0099,0.0099,0.0099,0.0099,0.0099,0.0099,0.0099,0.0099,...,0.0398,0.0,0.0398,0.0396,0.0099,0.0399,0.0398,0.0099,0.0099,0.0099


### Removing empty records

- We remove the 65 rows that hold only `customer_id` and the target: there is nothing to learn from them, and test has
  no such rows, so no rule is needed at prediction time (checked below).

In [6]:
empty = train_raw.drop(columns=["customer_id", target]).isna().all(axis=1)
empty_test = test_raw.drop(columns=["customer_id"]).isna().all(axis=1)
train = train_raw.loc[~empty].copy()

print(f"empty rows removed from train: {empty.sum()} | empty rows in test: {empty_test.sum()}")
print(f"repeat rate in the empty rows {train_raw.loc[empty, target].mean():.3f} vs {train[target].mean():.3f} in the rest")
print(f"duplicated rows left in train: {train.duplicated().sum()}")
print(f"train now: {train.shape}")

empty rows removed from train: 65 | empty rows in test: 0
repeat rate in the empty rows 0.508 vs 0.560 in the rest
duplicated rows left in train: 0
train now: (6469, 38)


- The duplicated rows were also part of these 65 empty rows.

### Impossible and Missing Values

In [7]:
sentinel = 1e9
col_impossible = ["days_since_previous_purchase", "prior_mean_gap_days",   # columns with impossible values (EDA)
                  "prior_mean_purchase_value", "spend_365d"]


def clean_data(df):
    df = df.copy()
    # removes all negative and sentinel values
    for col in col_impossible:
        incorrect = (df[col] < 0) | (df[col] >= sentinel)
        df.loc[incorrect, col] = np.nan

    # prior_max_purchase_value can never be higher than prior_total_spend
    incorrect = df["prior_max_purchase_value"] > df["prior_total_spend"] + 0.01
    df.loc[incorrect, "prior_max_purchase_value"] = np.nan

    # prior_mean_purchase_value is recomputed from count and total wherever the customer has a history
    count = df["prior_purchase_count"]
    has_prior = count > 0
    df["prior_mean_purchase_value"] = df["prior_total_spend"] / count.where(has_prior)

    # with exactly one prior purchase, days since that purchase = customer tenure (checked below)
    is_to_fill = (count == 1) & df["days_since_previous_purchase"].isna()
    df.loc[is_to_fill, "days_since_previous_purchase"] = df.loc[is_to_fill, "customer_tenure_days"]
    return df


def check_miss_imp(df_train, df_test):
    numeric = [c for c in df_train.select_dtypes(include="number").columns if c != target]
    found = False
    for column in [c for c in numeric if c not in ("customer_id", "random_noise")]:
        flags = {split: {"== 1e9": int((frame[column] >= 1e9).sum()),
                         "< 0": int((frame[column] < 0).sum())}
                 for split, frame in [("train", df_train), ("test", df_test)]}
        if any(flags["train"].values()) or any(flags["test"].values()):
            print(f"{column:30s} train {flags['train']}  test {flags['test']}")
            found = True
    for split, frame in [("train", df_train), ("test", df_test)]:
        n = int((frame["prior_max_purchase_value"] > frame["prior_total_spend"] + 0.01).sum())
        if n:
            print(f"prior_max > prior_total_spend   {split}: {n}")
            found = True
    if not found:
        print("no impossible values left in train or test")

In [8]:
check_miss_imp(train, test_raw)

days_since_previous_purchase   train {'== 1e9': 0, '< 0': 16}  test {'== 1e9': 0, '< 0': 2}
prior_mean_purchase_value      train {'== 1e9': 16, '< 0': 16}  test {'== 1e9': 2, '< 0': 2}
prior_mean_gap_days            train {'== 1e9': 0, '< 0': 16}  test {'== 1e9': 0, '< 0': 2}
spend_365d                     train {'== 1e9': 15, '< 0': 15}  test {'== 1e9': 2, '< 0': 2}
prior_max > prior_total_spend   train: 45
prior_max > prior_total_spend   test: 6


- The same errors exist in **test**, so rows cannot simply be dropped: the cleaning must be a rule applied to both files.
- Justification of the two repair rules (computed on the raw values, before cleaning):

In [9]:
valid = train.loc[train["prior_purchase_count"] == 1]
valid = valid[valid["days_since_previous_purchase"].between(0, sentinel, inclusive="left")]
print(f"one prior purchase, valid days_since: {len(valid)} rows | days_since == tenure in "
      f"{(valid['days_since_previous_purchase'] == valid['customer_tenure_days']).mean():.1%} of them")

ok = train["prior_mean_purchase_value"].between(0, sentinel, inclusive="neither") & (train["prior_purchase_count"] > 0)
gap = (train.loc[ok, "prior_mean_purchase_value"]
       - train.loc[ok, "prior_total_spend"] / train.loc[ok, "prior_purchase_count"]).abs()
print(f"valid prior means equal to total / count (within 1 cent): {(gap < 0.01).mean():.1%}")

one prior purchase, valid days_since: 423 rows | days_since == tenure in 100.0% of them
valid prior means equal to total / count (within 1 cent): 99.2%


In [10]:
train = clean_data(train)
test = clean_data(test_raw)
check_miss_imp(train, test)

no impossible values left in train or test


# Feature Engineering

From the EDA we know that repeat purchase is driven mostly by recency and frequency, so we create features that express
these factors:
- whether the customer has bought before (`has_purchased_before`), and whether there are enough purchases for a standard
  deviation to exist (`has_2plus_purchases`, `has_3plus_purchases`): these explain the structural gaps in the `prior_*`
  columns;
- recency: any purchase in the last 30 and 90 days;
- frequency: purchases per year of tenure;
- the share of the year's purchases that are recent;
- the discount rate of the current purchase.

All of them use one row at a time, so applying them to train and test cannot leak.

In [11]:
new_features = ["has_purchased_before", "has_2plus_purchases", "has_3plus_purchases", "bought_30d", "bought_90d",
                "purchases_per_year", "recent_share_30d", "recent_share_90d", "discount_rate"]


def add_features(df, share_fill=0.0, discount="line+global"):
    df = df.copy()
    count = df["prior_purchase_count"]

    df["has_purchased_before"] = (count >= 1).astype(int)
    df["has_2plus_purchases"] = (count >= 2).astype(int)
    df["has_3plus_purchases"] = (count >= 3).astype(int)
    df["bought_30d"] = (df["purchase_count_30d"] > 0).astype(int)
    df["bought_90d"] = (df["purchase_count_90d"] > 0).astype(int)
    df["purchases_per_year"] = count / (df["customer_tenure_days"] + 1) * 365
    # no purchase in the last 365 days -> the recent share is 0 by definition (tested below)
    df["recent_share_30d"] = (df["purchase_count_30d"] / df["purchase_count_365d"]).fillna(share_fill)
    df["recent_share_90d"] = (df["purchase_count_90d"] / df["purchase_count_365d"]).fillna(share_fill)
    discount_amount = df["Total Desc_ Linha"] + (df["Total Desc_ Global"] if discount == "line+global" else 0)
    df["discount_rate"] = discount_amount / df["Total Bruto"]
    return df


train = add_features(train)
test = add_features(test)

check = train[new_features + ["random_noise"]].corrwith(train[target], method="spearman")
print(check.sort_values(key=abs, ascending=False).round(3).to_string())
print("\nmissing / infinite values in the new features (train, test):",
      int(train[new_features].isna().sum().sum()), int(test[new_features].isna().sum().sum()), "/",
      int(np.isinf(train[new_features]).sum().sum()), int(np.isinf(test[new_features]).sum().sum()))

purchases_per_year      0.420
bought_90d              0.327
recent_share_90d        0.237
bought_30d              0.234
discount_rate           0.218
has_3plus_purchases     0.209
has_2plus_purchases     0.186
recent_share_30d        0.183
has_purchased_before    0.146
random_noise           -0.008

missing / infinite values in the new features (train, test): 0 0 / 0 0


- Every new feature is more correlated with the target than `random_noise`. This is a univariate screen only: whether the
  model gains from them is measured below, on the folds.

# clientes.csv

In [12]:
# Only the columns worth testing (EDA 1.9); the rest are empty or act as identifiers
client_cols = ["country_or_region", "commercial_region", "customer_since_year"]
clients_small = clientes_raw.set_index("customer_id")[client_cols]


def add_clientes(df):
    df = df.join(clients_small, on="customer_id")
    df["customer_age_years"] = df["purchase_date"].dt.year - df["customer_since_year"]
    return df


train = add_clientes(train)
test = add_clientes(test)
print("purchases dated before the customer's start year (train / test):",
      int((train["customer_age_years"] < 0).sum()), "/", int((test["customer_age_years"] < 0).sum()))
print("Spearman customer_age_years vs customer_tenure_days:",
      round(train["customer_age_years"].corr(train["customer_tenure_days"], method="spearman"), 3))

purchases dated before the customer's start year (train / test): 0 / 0
Spearman customer_age_years vs customer_tenure_days: 0.917


# Data Split

- `X_train` keeps **every candidate column**; which ones the model uses is decided by the column lists of the
  preprocessor, measured in the next section. At the end `X_train` and `X_test` are cut down to the columns used.
- `purchase_date` is kept aside: the time split needs it, the model does not (test dates lie beyond train, EDA 1.8).

In [13]:
dates_train = train["purchase_date"]
y = train[target]
X_train = train.drop(columns=[target, "purchase_date"])
X_test = test[X_train.columns]
print("X:", X_train.shape, " X_test:", X_test.shape)


def time_splits_with_horizon(dates, n_splits=5, horizon_days=90):
    """TimeSeriesSplit on date-sorted rows, dropping the training rows whose 90-day label reaches the validation fold."""
    for tr_idx, va_idx in TimeSeriesSplit(n_splits=n_splits).split(dates):
        va_start = dates.iloc[va_idx[0]]
        keep = dates.iloc[tr_idx] < va_start - pd.Timedelta(days=horizon_days)
        yield tr_idx[keep.to_numpy()], va_idx


cross_v = list(time_splits_with_horizon(dates_train))

folds = pd.DataFrame([{"train rows": len(tr), "train ends": dates_train.iloc[tr].max().date(),
                       "valid starts": dates_train.iloc[va].min().date(), "valid ends": dates_train.iloc[va].max().date(),
                       "valid rows": len(va), "valid repeat rate": round(y.iloc[va].mean(), 3)}
                      for tr, va in cross_v])
print(folds.to_string())

X: (6469, 49)  X_test: (986, 49)
   train rows  train ends valid starts  valid ends  valid rows  valid repeat rate
0         940  2017-05-12   2017-08-12  2019-01-12        1078              0.584
1        1922  2018-10-13   2019-01-12  2020-10-03        1078              0.534
2        3109  2020-07-04   2020-10-03  2022-03-05        1078              0.576
3        4189  2021-12-04   2022-03-05  2023-09-25        1078              0.520
4        5207  2023-06-24   2023-09-25  2025-02-15        1078              0.604


# Measuring the Choices

`make_preprocessor(opts)` builds the preprocessing from a dictionary of choices; `compare` scores several versions on the
same folds and reads the paired differences against a reference. Everything fitted (imputer, log, scaler, encoder) is
refitted on the training part of each fold.

Verdict: **better / worse** when the mean paired gain is larger than 2 standard errors, otherwise **inside the noise**.

In [14]:
IMPUTERS = {
    "median":     lambda: SimpleImputer(strategy="median"),
    "mean":       lambda: SimpleImputer(strategy="mean"),
    "median+ind": lambda: SimpleImputer(strategy="median", add_indicator=True),
    "knn":        lambda: KNNImputer(n_neighbors=15, weights="distance"),
    "iterative":  lambda: IterativeImputer(max_iter=10, initial_strategy="median", random_state=RANDOM_STATE),
}


def numeric_pipe(imputer, log):
    steps = [("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))] if log else []
    # log first, then impute: identical for the median (log keeps the order), and KNN/iterative work on the log scale
    return Pipeline(steps + [("impute", IMPUTERS[imputer]()), ("scale", StandardScaler())])


def category_pipe(min_rows, fill):
    imputer = (SimpleImputer(strategy="most_frequent") if fill == "mode"
               else SimpleImputer(strategy="constant", fill_value="missing"))
    return Pipeline([("impute", imputer),
                     ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",   # unseen test levels -> rare group
                                              min_frequency=min_rows, sparse_output=False))])


def make_preprocessor(o):
    blocks = [("skewed", numeric_pipe(o["imputer"], log=True), o["skewed"]),
              ("plain", numeric_pipe(o["imputer"], log=False), o["plain"]),
              ("flags", "passthrough", o["flags"]),
              ("low_cat", category_pipe(o["low_min"], o["cat_fill"]), o["low_cat"]),
              ("transport", category_pipe(o["high_min"], o["cat_fill"]), o["high_cat"])]
    return ColumnTransformer([b for b in blocks if len(b[2])], remainder="drop")


def cv_auc(o, X=None):
    X = X_train if X is None else X
    scores = []
    for tr, va in cross_v:
        model = Pipeline([("prep", make_preprocessor(o)), ("clf", LogisticRegression(max_iter=5000))])
        model.fit(X.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[va], model.predict_proba(X.iloc[va])[:, 1]))
    return np.array(scores)


def compare(variants, reference):
    """variants: {name: opts} or {name: (opts, X)}. Returns one row per variant, paired against `reference`."""
    runs = {name: cv_auc(*(v if isinstance(v, tuple) else (v,))) for name, v in variants.items()}
    ref, rows = runs[reference], []
    for name, s in runs.items():
        d = s - ref
        sem = d.std(ddof=1) / np.sqrt(len(d))
        verdict = ("reference" if name == reference else
                   ("better" if d.mean() > 0 else "worse") if abs(d.mean()) > 2 * sem else "inside the noise")
        rows.append({"version": name, "AUC": s.mean(), "AUC last 2 folds": s[-2:].mean(), "dAUC": d.mean(),
                     "SEM": sem, "wins": f"{(d > 0).sum()}/{len(d)}", "verdict": verdict})
    table = pd.DataFrame(rows).set_index("version")
    print(table.to_string(formatters={"AUC": "{:.4f}".format, "AUC last 2 folds": "{:.4f}".format,
                                      "dAUC": "{:+.4f}".format, "SEM": "{:.4f}".format}))
    return table


DECISIONS = []


def decide(step, choice, evidence):
    DECISIONS.append({"step": step, "decision": choice, "evidence": evidence})
    print(f"DECISION {step}: {choice}")


# Starting point = the group's original recipe
flag_cols = ["has_purchased_before", "bought_30d", "bought_90d"]
low_cat_cols = ["payment_type", "salesperson", "country_or_region", "commercial_region"]
high_cat_cols = ["transport_method"]
plain_num_cols = ["customer_tenure_days", "customer_age_years", "discount_rate",
                  "recent_share_30d", "recent_share_90d", "random_noise"]
NOT_FEATURES = ["customer_id", "commercial_zone", "warehouse", "document_series", "customer_since_year",
                "has_2plus_purchases", "has_3plus_purchases"]
skewed_num_cols = [c for c in X_train.columns
                   if c not in flag_cols + low_cat_cols + high_cat_cols + plain_num_cols + NOT_FEATURES]

opts = {"skewed": skewed_num_cols, "plain": plain_num_cols, "flags": flag_cols, "low_cat": low_cat_cols,
        "high_cat": high_cat_cols, "imputer": "median", "cat_fill": "constant", "low_min": 20, "high_min": 50}


def with_(o, **changes):
    return {**o, **changes}


def without(cols, *names):
    return [c for c in cols if c not in names]

### 1. Identifier-like columns

`customer_id` is a label, not a measurement (EDA 1.7), and `random_noise` is, by its name and its near-zero correlation,
noise. We check that neither carries usable signal.

In [15]:
board = compare({
    "start (random_noise in, customer_id out)": opts,
    "- random_noise": with_(opts, plain=without(opts["plain"], "random_noise")),
    "+ customer_id": with_(opts, plain=opts["plain"] + ["customer_id"]),
}, "start (random_noise in, customer_id out)")

                                            AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                               
start (random_noise in, customer_id out) 0.7732           0.7820 +0.0000 0.0000  0/5         reference
- random_noise                           0.7735           0.7822 +0.0003 0.0004  3/5  inside the noise
+ customer_id                            0.7726           0.7817 -0.0006 0.0008  3/5  inside the noise


- Removing `random_noise` gains **+0.0003 ± 0.0004** and adding `customer_id` loses **-0.0006 ± 0.0008**: neither column
  carries usable signal, as the EDA expected;
- Both are excluded. The brief asks for identifier-like variables to be treated critically: `customer_id` is a label,
  and a model that used it would not generalise to new customers; `random_noise` is noise by construction.

In [16]:
opts = with_(opts, plain=without(opts["plain"], "random_noise"))
decide("1 identifiers", "exclude customer_id and random_noise",
       "EDA 1.7-1.8 and comparison 1: neither adds signal")

DECISION 1 identifiers: exclude customer_id and random_noise


### 2. Categorical columns left out at the start

`commercial_zone` and `warehouse` are almost constant in train (EDA 1.3); `document_series` was left out because its
levels have similar repeat rates (EDA 1.11). We check each one, and how much the near-constant ones shift in test.

In [17]:
for col in ["commercial_zone", "warehouse", "document_series"]:
    top = X_train[col].value_counts(normalize=True).index[0]
    print(f"{col:16s} main level: {X_train[col].eq(top).mean():.1%} of train, {X_test[col].eq(top).mean():.1%} of test"
          f" | test levels never seen in train: {int((~X_test[col].isin(X_train[col].dropna().unique()) & X_test[col].notna()).sum())} rows")
print()
board = compare({
    "current": opts,
    "+ document_series": with_(opts, low_cat=opts["low_cat"] + ["document_series"]),
    "+ commercial_zone": with_(opts, low_cat=opts["low_cat"] + ["commercial_zone"]),
    "+ warehouse": with_(opts, low_cat=opts["low_cat"] + ["warehouse"]),
}, "current")

commercial_zone  main level: 96.8% of train, 70.1% of test | test levels never seen in train: 209 rows
warehouse        main level: 96.7% of train, 96.2% of test | test levels never seen in train: 7 rows
document_series  main level: 42.3% of train, 100.0% of test | test levels never seen in train: 0 rows



                     AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                        
current           0.7735           0.7822 +0.0000 0.0000  0/5         reference
+ document_series 0.7744           0.7829 +0.0010 0.0004  4/5            better
+ commercial_zone 0.7736           0.7821 +0.0001 0.0002  4/5  inside the noise
+ warehouse       0.7742           0.7832 +0.0008 0.0004  4/5  inside the noise


- `document_series` is the only column that improves the folds (**+0.0010 ± 0.0004**, 4/5). But in **test every row has
  the same level** (100.0%). A column that is constant in test cannot rank test customers: the gain seen in validation
  cannot reach the submission, and the column would only shift every test probability by the same amount, which moves
  the threshold. It stays out, now for this reason rather than the EDA one;
- `commercial_zone` adds nothing (+0.0001) and shifts between files: its main level covers 96.8% of train but only 70.1%
  of test, with 209 test rows on a level never seen in train;
- `warehouse` is inside the noise (+0.0008 ± 0.0004) and almost constant in both files. The simpler option is kept.

In [18]:
# document_series is NOT added: every test row has the same level (table above), so it cannot rank test customers
decide("2 categoricals", "leave out document_series, commercial_zone and warehouse",
       "EDA 1.3, 1.11; train/test shares (document_series is constant in test, commercial_zone shifts 96.8% -> 70.1%)"
       " and comparison 2")

DECISION 2 categoricals: leave out document_series, commercial_zone and warehouse


### 3. clientes.csv columns

The join is safe (no purchase before the customer's start year, printed above). We check whether the three columns add
anything once the purchase history is in the model.

In [19]:
no_cli = with_(opts, low_cat=without(opts["low_cat"], "country_or_region", "commercial_region"),
               plain=without(opts["plain"], "customer_age_years"))
board = compare({
    "current (all clientes columns)": opts,
    "no clientes columns": no_cli,
    "only customer_age_years": with_(no_cli, plain=no_cli["plain"] + ["customer_age_years"]),
    "only the two regions": with_(no_cli, low_cat=no_cli["low_cat"] + ["country_or_region", "commercial_region"]),
}, "current (all clientes columns)")

                                  AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                     
current (all clientes columns) 0.7735           0.7822 +0.0000 0.0000  0/5         reference
no clientes columns            0.7739           0.7823 +0.0004 0.0011  3/5  inside the noise
only customer_age_years        0.7738           0.7827 +0.0003 0.0007  3/5  inside the noise
only the two regions           0.7735           0.7820 +0.0001 0.0008  3/5  inside the noise


- No version improves the score: without the `clientes` columns **+0.0004 ± 0.0011**, the regions alone +0.0001, the
  age alone +0.0003, all inside the noise;
- `customer_age_years` correlates **0.917** with `customer_tenure_days`, which the model already has: it repeats the
  purchase history at a coarser (yearly) resolution;
- `clientes.csv` is therefore not used. The join stays in the notebook, so the modelling notebook can test it again
  with a non-linear model.

In [20]:
opts = no_cli
decide("3 clientes.csv", "not used: no column improves the score; customer_age_years repeats customer_tenure_days",
       "join check (rho 0.917 with tenure) and comparison 3")

DECISION 3 clientes.csv: not used: no column improves the score; customer_age_years repeats customer_tenure_days


### 4. Which numeric columns get `log1p`

The split into "skewed" and "plain" columns is based on skewness (EDA 1.10). The table shows the skew of every numeric
input and checks that no value is below 0, where `log1p` would fail; then we compare the split with logging everything
and logging nothing.

In [21]:
num_inputs = opts["skewed"] + opts["plain"]
skew = pd.DataFrame({"skew": X_train[num_inputs].skew(), "min": X_train[num_inputs].min(),
                     "group": ["skewed" if c in opts["skewed"] else "plain" for c in num_inputs]})
print(skew.sort_values("skew", ascending=False).round(2).to_string())
print("\nnegative values in the logged columns (train / test):",
      int((X_train[opts["skewed"]] < 0).sum().sum()), "/", int((X_test[opts["skewed"]] < 0).sum().sum()))
print()
board = compare({
    "log the skewed group (current)": opts,
    "log nothing": with_(opts, skewed=[], plain=num_inputs),
    "log every numeric column": with_(opts, skewed=num_inputs, plain=[]),
}, "log the skewed group (current)")

                                 skew   min   group
Total Desc_ Global              27.46  0.00  skewed
Total Portes                    19.55  0.00  skewed
spend_365d                      18.55  0.00  skewed
prior_mean_gap_days              9.31  1.00  skewed
prior_std_gap_days               7.77  0.00  skewed
days_since_previous_purchase     6.62  1.00  skewed
invoice_count_current_purchase   6.44  1.00  skewed
Total Desc_ Linha                5.92  0.00  skewed
prior_min_purchase_value         5.90  0.93  skewed
spend_30d                        5.50  0.00  skewed
Total Imp_                       4.84  0.00  skewed
Custo                            4.61  0.00  skewed
purchases_per_year               4.50  0.00  skewed
Total Bruto                      4.31  0.83  skewed
Total a Pagar                    4.21  0.93  skewed
Total Liquido                    4.15  0.76  skewed
prior_max_purchase_value         3.78  3.31  skewed
spend_90d                        3.72  0.00  skewed
prior_purcha

                                  AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                     
log the skewed group (current) 0.7739           0.7823 +0.0000 0.0000  0/5         reference
log nothing                    0.7690           0.7783 -0.0049 0.0036  1/5  inside the noise
log every numeric column       0.7722           0.7831 -0.0016 0.0020  2/5  inside the noise


- No option is measurably better, but the direction is consistent: logging nothing costs **-0.0049 ± 0.0036** (worse on
  4 of 5 folds), logging everything **-0.0016 ± 0.0020**;
- The decision rests on the skew table: the skewed group goes up to skew **27** and has no negative value in train or
  test, so `log1p` is safe there. On the original scale a few very large purchases would dominate a linear model;
- `recent_share_30d` (skew 3.1) is in the plain group: it is a share bounded by 0 and 1, so `log1p` would barely change it.

In [22]:
decide("4 log1p", "log1p on the skewed group (skew above 2, all values >= 0)",
       "EDA 1.10, skew table and comparison 4: no option measurably better, skew up to 27 decides")

DECISION 4 log1p: log1p on the skewed group (skew above 2, all values >= 0)


### 5. Missing numeric values

Most gaps in the `prior_*` columns are **structural**: the customer has too few purchases for the statistic to exist
(EDA 1.5). We compare imputers, missing-value indicators, and the two extra history flags that say "fewer than 2" and
"fewer than 3" purchases.

In [23]:
print(X_train.assign(n=X_train["prior_purchase_count"].clip(upper=3))
      .groupby("n")[["prior_std_purchase_value", "prior_mean_gap_days", "prior_std_gap_days"]]
      .agg(lambda s: s.isna().mean()).round(3).rename(index={3: "3+"}).to_string())
print()
more_flags = with_(opts, flags=opts["flags"] + ["has_2plus_purchases", "has_3plus_purchases"])
board = compare({
    "median (current)": opts,
    "mean": with_(opts, imputer="mean"),
    "median + indicators": with_(opts, imputer="median+ind"),
    "KNN (15 neighbours)": with_(opts, imputer="knn"),
    "iterative": with_(opts, imputer="iterative"),
    "median + 2+/3+ history flags": more_flags,
}, "median (current)")

     prior_std_purchase_value  prior_mean_gap_days  prior_std_gap_days
n                                                                     
0.0                       1.0                1.000                 1.0
1.0                       1.0                1.000                 1.0
2.0                       0.0                0.068                 1.0
3+                        0.0                0.037                 0.0



                                AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                   
median (current)             0.7739           0.7823 +0.0000 0.0000  0/5         reference
mean                         0.7739           0.7823 +0.0000 0.0001  3/5  inside the noise
median + indicators          0.7734           0.7815 -0.0004 0.0006  2/5  inside the noise
KNN (15 neighbours)          0.7742           0.7828 +0.0003 0.0005  3/5  inside the noise
iterative                    0.7710           0.7837 -0.0028 0.0021  2/5  inside the noise
median + 2+/3+ history flags 0.7740           0.7823 +0.0002 0.0004  3/5  inside the noise


- The table confirms the gaps are **structural**: `prior_std_purchase_value` is missing for every customer with fewer than
  2 prior purchases and never otherwise, and `prior_std_gap_days` for every customer with fewer than 3;
- No imputer is measurably better than the median: mean +0.0000, KNN +0.0003, iterative -0.0028, indicators -0.0004,
  the extra 2+/3+ flags +0.0002, all inside the noise. The `has_purchased_before` flag already tells the model when the
  history is empty;
- The **median** is kept: the simplest option, robust to the long tails of EDA 1.10, and fitted on the training fold only.

In [24]:
decide("5 missing values", "median imputation, fitted in the fold; no indicators, no extra flags",
       "EDA 1.5, gap table and comparison 5: no imputer measurably better, the median is the simplest and robust to tails")

DECISION 5 missing values: median imputation, fitted in the fold; no indicators, no extra flags


### 6. Missing categories

Two options: a separate `"missing"` level, or the most frequent level. If missing categories repeat at about the
overall rate, a separate level has nothing to learn.

In [25]:
cat_now = opts["low_cat"] + opts["high_cat"]
print(pd.DataFrame({c: {"missing rows": int(X_train[c].isna().sum()),
                        "repeat rate if missing": round(y[X_train[c].isna()].mean(), 3)} for c in cat_now}).T.to_string())
print(f"overall repeat rate: {y.mean():.3f}\n")
board = compare({"'missing' level (current)": opts, "most frequent level": with_(opts, cat_fill="mode")},
                "'missing' level (current)")

                  missing rows  repeat rate if missing
payment_type             195.0                   0.564
salesperson              194.0                   0.598
transport_method         196.0                   0.551
overall repeat rate: 0.560



                             AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                
'missing' level (current) 0.7739           0.7823 +0.0000 0.0000  0/5         reference
most frequent level       0.7754           0.7824 +0.0016 0.0014  3/5  inside the noise


- Rows with a missing category repeat at **0.551-0.598**, close to the overall **0.560**: a separate level has little to
  learn;
- The comparison agrees: the most frequent level scores **+0.0016 ± 0.0014**, inside the noise;
- On a tie the simpler option wins: filling with the most frequent level adds no extra column.

In [26]:
opts = with_(opts, cat_fill="mode")
decide("6 missing categories", "fill with the most frequent level",
       "repeat rates of missing rows close to overall, and comparison 6 (tie -> no extra column)")

DECISION 6 missing categories: fill with the most frequent level


### 7. Rare levels

Levels with fewer rows than `min_frequency` are grouped into one "infrequent" column; unseen test levels go there too.
`transport_method` has 80 levels, many of them rare (EDA 1.3).

In [27]:
for col, m in [("payment_type", opts["low_min"]), ("transport_method", opts["high_min"])]:
    counts = X_train[col].value_counts()
    print(f"{col:16s} {len(counts)} levels | kept with min_frequency={m}: {(counts >= m).sum()} | "
          f"with 10: {(counts >= 10).sum()}")
print()
board = compare({
    "20 / 50 (current)": opts,
    "10 / 10": with_(opts, low_min=10, high_min=10),
    "10 / 20": with_(opts, low_min=10, high_min=20),
    "50 / 100": with_(opts, low_min=50, high_min=100),
}, "20 / 50 (current)")

payment_type     9 levels | kept with min_frequency=20: 6 | with 10: 6
transport_method 80 levels | kept with min_frequency=50: 23 | with 10: 40



                     AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                        
20 / 50 (current) 0.7754           0.7824 +0.0000 0.0000  0/5         reference
10 / 10           0.7781           0.7859 +0.0027 0.0019  3/5  inside the noise
10 / 20           0.7780           0.7859 +0.0025 0.0021  3/5  inside the noise
50 / 100          0.7836           0.7840 +0.0082 0.0047  4/5  inside the noise


- `transport_method` keeps 23 of its 80 levels with 50 rows, 40 with 10; `payment_type` keeps 6 of 9 in both cases;
- No threshold beats 20 / 50 measurably. The largest gain, 50 / 100 (**+0.0082 ± 0.0047**), comes from the small early
  folds: on the last two folds, the most test-like, it is behind 10 / 10 (0.7840 against 0.7859). With no clear winner,
  the original thresholds are kept;
- Unseen test levels go to the infrequent group, so the test file transforms without error.

In [28]:
decide("7 rare levels", "min_frequency 20 (payment, salesperson) / 50 (transport_method)",
       "EDA 1.3, level counts and comparison 7: no threshold measurably better, the original is kept")

DECISION 7 rare levels: min_frequency 20 (payment, salesperson) / 50 (transport_method)


### 8. Definitions of two engineered features

- `recent_share_*` is undefined when the customer has no purchase in the last 365 days. By definition the recent share
  is then 0; the alternative is to leave it missing and let the median fill it.
- `discount_rate` can count only the line discount, or the line plus the global discount.

In [29]:
undefined = train["purchase_count_365d"] == 0
print(f"rows with no purchase in 365 days: {undefined.sum()} | repeat rate {y[undefined].mean():.3f} vs {y.mean():.3f}")
print("rows with a global discount and no line discount:",
      int(((train["Total Desc_ Global"] > 0) & (train["Total Desc_ Linha"] == 0)).sum()))
print()
cols = X_train.columns


def rebuild(**kw):
    frame = add_clientes(add_features(clean_data(train_raw.loc[~empty]), **kw))
    return frame.loc[X_train.index, cols]


board = compare({
    "share = 0, line + global discount (current)": opts,
    "share left missing (median fill)": (opts, rebuild(share_fill=np.nan)),
    "line discount only": (opts, rebuild(discount="line")),
}, "share = 0, line + global discount (current)")

rows with no purchase in 365 days: 843 | repeat rate 0.320 vs 0.560
rows with a global discount and no line discount: 37



                                               AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                                  
share = 0, line + global discount (current) 0.7754           0.7824 +0.0000 0.0000  0/5         reference
share left missing (median fill)            0.7756           0.7821 +0.0002 0.0003  2/5  inside the noise
line discount only                          0.7755           0.7822 +0.0001 0.0001  3/5  inside the noise


- The 843 customers with no purchase in the last 365 days repeat at only **0.320**. Filling their recent share with 0 or
  with the median scores the same (+0.0002 ± 0.0003), so the choice is made on meaning: with no purchase in the year, the
  share of recent purchases **is** 0, and a fixed rule needs nothing fitted;
- Counting the global discount changes nothing measurable (+0.0001); it is kept because it is what a discount rate
  means, and 37 purchases have a global discount and no line discount.

In [30]:
decide("8 feature definitions", "recent share = 0 when undefined; discount = line + global", "comparison 8")

DECISION 8 feature definitions: recent share = 0 when undefined; discount = line + global


### 9. Do the engineered features help the model?

Finally, the recipe with and without the engineered features (the flags stay: they describe missingness, step 5).

In [31]:
engineered = ["purchases_per_year", "recent_share_30d", "recent_share_90d", "discount_rate", "bought_30d", "bought_90d"]
base_nf = with_(opts, skewed=without(opts["skewed"], *engineered), plain=without(opts["plain"], *engineered),
                flags=without(opts["flags"], *engineered))
variants = {"with engineered features (current)": opts, "without any": base_nf}
for f in engineered:
    variants[f"without {f}"] = with_(opts, skewed=without(opts["skewed"], f), plain=without(opts["plain"], f),
                                     flags=without(opts["flags"], f))
board = compare(variants, "with engineered features (current)")

                                      AUC AUC last 2 folds    dAUC    SEM wins           verdict
version                                                                                         
with engineered features (current) 0.7754           0.7824 +0.0000 0.0000  0/5         reference
without any                        0.7744           0.7830 -0.0010 0.0015  3/5  inside the noise
without purchases_per_year         0.7739           0.7823 -0.0016 0.0006  1/5             worse
without recent_share_30d           0.7751           0.7825 -0.0003 0.0002  1/5  inside the noise
without recent_share_90d           0.7755           0.7821 +0.0001 0.0003  2/5  inside the noise
without discount_rate              0.7756           0.7822 +0.0002 0.0002  4/5  inside the noise
without bought_30d                 0.7756           0.7823 +0.0002 0.0002  3/5  inside the noise
without bought_90d                 0.7764           0.7826 +0.0009 0.0004  4/5            better


- `purchases_per_year` is the one engineered feature the model measurably needs: removing it costs **-0.0016 ± 0.0006**;
- `bought_90d` measurably hurts: removing it gains **+0.0009 ± 0.0004** (4/5). It repeats `purchase_count_90d`, whose
  `log1p` already separates zero from non-zero;
- The other four (`bought_30d`, `recent_share_30d`, `recent_share_90d`, `discount_rate`) are inside the noise. They stay
  in for now, marked for the feature-selection step, which judges them alongside every other column.

In [32]:
opts = with_(opts, flags=without(opts["flags"], "bought_90d"))
decide("9 engineered features",
       "drop bought_90d (measured worse); keep purchases_per_year (measured better); the other four are passed to feature selection",
       "comparison 9")

DECISION 9 engineered features: drop bought_90d (measured worse); keep purchases_per_year (measured better); the other four are passed to feature selection


# Final Preprocessing Pipeline

The recipe below is the result of the decisions above. As a check only, it is fitted on train and applied to test; in
the modelling notebook it is refitted inside every fold.

In [33]:
preprocessor = make_preprocessor(opts).set_output(transform="pandas")
used_cols = opts["skewed"] + opts["plain"] + opts["flags"] + opts["low_cat"] + opts["high_cat"]
X_train = X_train[used_cols]
X_test = X_test[used_cols]

X_check = preprocessor.fit_transform(X_train)
X_test_check = preprocessor.transform(X_test)
print("train:", X_check.shape, " test:", X_test_check.shape)
print("missing values left (train / test):", int(X_check.isna().sum().sum()), "/", int(X_test_check.isna().sum().sum()))
print("same output columns:", list(X_check.columns) == list(X_test_check.columns))
assert X_test.index.tolist() == sample["ID"].tolist(), "X_test must follow sample_submission order"
print("X_test in sample_submission order: True")

print()
original = {"skewed": skewed_num_cols, "plain": plain_num_cols, "flags": flag_cols, "low_cat": low_cat_cols,
            "high_cat": high_cat_cols, "imputer": "median", "cat_fill": "constant", "low_min": 20, "high_min": 50}
X_train_all = train.drop(columns=[target, "purchase_date"])
board = compare({"original recipe": (original, X_train_all), "final recipe": (opts, X_train_all)}, "original recipe")

print()
print(pd.DataFrame(DECISIONS).to_string(index=False))

train: (6469, 72)  test: (986, 72)
missing values left (train / test): 0 / 0
same output columns: True
X_test in sample_submission order: True



                   AUC AUC last 2 folds    dAUC    SEM wins    verdict
version                                                               
original recipe 0.7732           0.7820 +0.0000 0.0000  0/5  reference
final recipe    0.7764           0.7826 +0.0032 0.0011  5/5     better

                 step                                                                                                                    decision                                                                                                                       evidence
        1 identifiers                                                                                        exclude customer_id and random_noise                                                                              EDA 1.7-1.8 and comparison 1: neither adds signal
       2 categoricals                                                                    leave out document_series, commercial_zone and warehouse EDA 1.3, 1.11; train/te

- The final recipe scores **0.7764** AUC against **0.7732** for the original (**+0.0032 ± 0.0011**, better on **all 5
  folds**); on the last two folds, the most test-like, 0.7826 against 0.7820;
- Fitted on all of train it produces **72 columns** for train and test alike, with no missing value, and `X_test` follows
  the row order of `sample_submission.csv` (checked by an `assert`);
- Measured decisions: dropping `random_noise` and `customer_id` (1), `purchases_per_year` kept and `bought_90d` dropped
  (9). Decided on stated principles where the score could not separate the options: `document_series` constant in test
  and `commercial_zone` shifting (2), skew (4), simplest option on a tie (5, 6, 7), meaning of the feature (8);
- Hand-off: feature selection judges the four engineered features left inside the noise; the modelling notebook tunes
  the decision threshold.

# Handoff

In [34]:
import joblib

# Everything the feature-selection and modelling notebooks need, in one file
handoff = {
    "X_train": X_train,              # 6,469 rows, cleaned, with the features used by the recipe
    "y": y,                          # target, same row order as X_train
    "X_test": X_test,                # 986 rows, same columns, in sample_submission order
    "dates_train": dates_train,      # purchase dates of X_train, to rebuild time splits if needed
    "cross_v": cross_v,              # the 5 time-ordered folds with the 90-day gap
    "preprocessor": clone(preprocessor),   # the recipe, UNFITTED: each fold refits it
    "column_groups": {k: opts[k] for k in ("skewed", "plain", "flags", "low_cat", "high_cat")},
    "decisions": DECISIONS,
}

output_file = DATA_DIR / "preprocessed.joblib"
joblib.dump(handoff, output_file)
print("saved to", output_file)
print({name: getattr(obj, "shape", len(obj) if hasattr(obj, "__len__") else type(obj).__name__)
       for name, obj in handoff.items()})

saved to data/preprocessed.joblib
{'X_train': (6469, 37), 'y': (6469,), 'X_test': (986, 37), 'dates_train': (6469,), 'cross_v': 5, 'preprocessor': 'ColumnTransformer', 'column_groups': 5, 'decisions': 9}
